In [ ]:
import pandas as pd
import numpy as np
import joblib


#xgboost?
from xgboost import XGBClassifier
from sklearn.model_selection import LeaveOneGroupOut,GroupKFold
from sklearn.metrics import classification_report, accuracy_score


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler


# Ucitavanje GRAPE dataset

In [5]:
# Učitavanje Baseline podataka (početno stanje)
df_base = pd.read_excel("VF and clinical information.xlsx", sheet_name="Baseline")

# Učitavanje Follow-up podataka (praćenje kroz vrijeme)
df_follow = pd.read_excel("VF and clinical information.xlsx", sheet_name="Follow-up")

print("Baseline oblik:", df_base.shape)
print("Follow-up oblik:", df_follow.shape)

Baseline oblik: (264, 80)
Follow-up oblik: (1116, 69)


In [ ]:
df_base.info()

In [ ]:
df_follow.info()

# LSTM

In [11]:
# ==========================================
# 1. UCITAVANJE I REORGANIZACIJA PODATAKA (SIGURNA VERZIJA)
# ==========================================


# Preimenovanje kolona u Baseline
base_cols = list(df_base.columns)
base_cols[9] = 'Progression_MD'
base_cols[12] = 'RNFL_S'
base_cols[13] = 'RNFL_N'
base_cols[14] = 'RNFL_I'
base_cols[15] = 'RNFL_T'
for i in range(19, len(base_cols)): base_cols[i] = f'VF_{i-18}'
df_base.columns = base_cols

# Preimenovanje kolona u Follow-up
follow_cols = list(df_follow.columns)
for i in range(8, len(follow_cols)): follow_cols[i] = f'VF_{i-7}'
df_follow.columns = follow_cols

# Osiguravamo Laterality i Visit Number
df_base['Laterality'] = df_base['Laterality'].astype(str).str.strip().map({'OD': 1, 'OS': 0}).fillna(0).astype(int)
df_follow['Laterality'] = df_follow['Laterality'].astype(str).str.strip().map({'OD': 1, 'OS': 0}).fillna(0).astype(int)

df_base['Visit Number'] = 0
df_follow['Visit Number'] = pd.to_numeric(df_follow['Visit Number'], errors='coerce').fillna(0).astype(int)

# --- KLJUČNA IZMJENA: Razdvajamo selekciju jer Follow-up nema demografiju ---
vf_cols = [f'VF_{i}' for i in range(1, 62)]

# Iz Baseline-a uzimamo sve što nam treba
base_dynamic_cols = ['Subject Number', 'Laterality', 'Visit Number', 'IOP'] + vf_cols
df_b_dyn = df_base[base_dynamic_cols].copy()

# Iz Follow-up-a uzimamo SAMO ono što stvarno postoji u njemu
follow_dynamic_cols = ['Subject Number', 'Laterality', 'Visit Number', 'IOP'] + vf_cols
df_f_dyn = df_follow[follow_dynamic_cols].copy()

# Spajamo sve posjete hronološki (za sada imamo samo ID-eve, posjete, IOP i VF)
df_all = pd.concat([df_b_dyn, df_f_dyn], axis=0)
df_all = df_all.sort_values(by=['Subject Number', 'Laterality', 'Visit Number']).reset_index(drop=True)

# Sada izvlačimo STATIČKE podatke pacijenata iz Baseline-a (sa RNFL i demografijom)
df_static = df_base[['Subject Number', 'Laterality', 'Age', 'Gender', 'CCT', 'Category of Glaucoma', 'RNFL_S', 'RNFL_N', 'RNFL_I', 'RNFL_T']].copy()

# Mapiramo tekstualne kategorije u brojeve unutar statičkih podataka
df_static['Gender'] = df_static['Gender'].astype(str).str.strip().map({'M': 1, 'F': 0}).fillna(0).astype(int)
df_static['Category of Glaucoma'] = df_static['Category of Glaucoma'].astype(str).str.strip().map({'OAG': 1, 'ACG': 0}).fillna(0).astype(int)
for col in ['RNFL_S', 'RNFL_N', 'RNFL_I', 'RNFL_T']:
    df_static[col] = pd.to_numeric(df_static[col].replace('/', np.nan), errors='coerce').fillna(0)

# SPAJANJE: Sada radimo merge tako da svaka posjeta dobije pripadajuće statičke podatke pacijenta!
df_all = pd.merge(df_all, df_static, on=['Subject Number', 'Laterality'], how='left')

# Čišćenje IOP-a i računanje VF_mean nakon spajanja
df_all['IOP'] = pd.to_numeric(df_all['IOP'].replace('/', np.nan), errors='coerce')
df_all['IOP'] = df_all.groupby(['Subject Number', 'Laterality'])['IOP'].transform(lambda x: x.fillna(method='ffill')).fillna(0).astype(float)

for col in vf_cols:
    df_all[col] = pd.to_numeric(df_all[col], errors='coerce').fillna(0)
df_all['VF_mean'] = df_all[vf_cols].mean(axis=1)

# Definišemo konačne varijable koje LSTM gleda (Sve su garantovano tu i numeričke su)
features_list = ['Age', 'Gender', 'CCT', 'Category of Glaucoma', 'IOP', 'VF_mean', 'RNFL_S', 'RNFL_N', 'RNFL_I', 'RNFL_T']

for col in features_list:
    df_all[col] = df_all[col].astype(float)

# Skaliranje
scaler = StandardScaler()
df_all[features_list] = scaler.fit_transform(df_all[features_list])

# Target (Progression) i Grupe vezujemo za pacijenta (Sigurna verzija)
df_targets = df_base[['Subject Number', 'Laterality', 'Progression_MD']].copy()

# Prisilno pretvaramo u broj, tekst 'MD' postaje NaN, i onda ga menjamo sa 0
df_targets['Progression_MD'] = pd.to_numeric(df_targets['Progression_MD'], errors='coerce').fillna(0).astype(int)

# ==========================================
# 2. PAKOVANJE U 3D STRUKTURU (VREMENSKE SERIJE)
# ==========================================
# Pronalazimo koliko maksimalno posjeta ima neki pacijent u datasetu
max_visits = df_all.groupby(['Subject Number', 'Laterality']).size().max()

X_sequences = []
y_labels = []
patient_groups = []

# Prolazimo kroz svako oko svakog pacijenta i gradimo istorijski niz
for _, group in df_all.groupby(['Subject Number', 'Laterality']):
    sub_num = group['Subject Number'].iloc[0]
    lat = group['Laterality'].iloc[0]

    # Izvlačimo istoriju posjeta za ovo oko
    seq = group[features_list].values
    actual_length = len(seq)

    # PADDING: Ako pacijent ima manje posjeta od max_visits, dopunjavamo nulama na kraju
    # LSTM će naučiti da ignoriše ove nule
    if actual_length < max_visits:
        padding = np.zeros((max_visits - actual_length, len(features_list)))
        seq = np.vstack((seq, padding))

    # Izvlačimo istorijsku oznaku da li je pacijent na kraju progredirao
    target_row = df_targets[(df_targets['Subject Number'] == sub_num) & (df_targets['Laterality'] == lat)]
    if not target_row.empty:
        X_sequences.append(seq)
        y_labels.append(target_row['Progression_MD'].iloc[0])
        patient_groups.append(sub_num)

X_3D = np.array(X_sequences, dtype=np.float32)
y_3D = np.array(y_labels, dtype=np.float32)
groups_3D = np.array(patient_groups)

print(f"Podaci spakovani u 3D tenzor oblika: {X_3D.shape}") # (Pacijenti, Posjete, Karakteristike)

/tmp/ipykernel_9098/2397105096.py:57: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_all['IOP'] = df_all.groupby(['Subject Number', 'Laterality'])['IOP'].transform(lambda x: x.fillna(method='ffill')).fillna(0).astype(float)


Podaci spakovani u 3D tenzor oblika: (144, 40, 10)


In [12]:
#LSTM model
class GlaucomaLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(GlaucomaLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # LSTM Sloj
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)

        # Fully Connected (Gusti) izlazni sloj za binarnu klasifikaciju
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Inicijalizacija skrivenih stanja (hidden i cell state) sa nulama
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        # Prolazak kroz LSTM
        out, _ = self.lstm(x, (h0, c0))

        # Uzimamo izlaz samo iz POSLJEDNJEG vremenskog koraka (zadnje posjete)
        out = self.fc(out[:, -1, :])
        out = self.sigmoid(out)
        return out

In [13]:
#LSTM trening i validacija
gkf = GroupKFold(n_splits=5)
all_fold_accuracies = []

print("\n=== ZAPOČINJEMO TRENING LSTM MREŽE ===")

for fold, (train_idx, test_idx) in enumerate(gkf.split(X_3D, y_3D, groups=groups_3D)):
    # Podjela na train i test skupove
    X_train, X_test = torch.tensor(X_3D[train_idx]), torch.tensor(X_3D[test_idx])
    y_train, y_test = torch.tensor(y_3D[train_idx]).unsqueeze(1), torch.tensor(y_3D[test_idx]).unsqueeze(1)

    # Pakovanje u PyTorch DataLoadere
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=16, shuffle=True)

    # Inicijalizacija modela, funkcije gubitka (Loss) i optimizatora
    model = GlaucomaLSTM(input_size=len(features_list), hidden_size=32, num_layers=2)
    criterion = nn.BCELoss() # Binary Cross Entropy za 0/1 klasifikaciju
    optimizer = optim.Adam(model.parameters(), lr=0.005)

    # Trening petlja kroz epohe
    model.train()
    for epoch in range(40): # 40 epoha je sasvim dovoljno za ovaj obim podataka
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    # Evaluacija na test fold-u (neviđenim pacijentima)
    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test)
        # Ako je vjerovatnoća > 0.5, klasifikujemo kao 1 (Progresija)
        preds = (test_outputs >= 0.5).float()
        acc = (preds == y_test).float().mean().item()
        all_fold_accuracies.append(acc)

    print(f"Fold {fold + 1} Završen | Trenutni Loss: {loss.item():.4f} | Tačnost na test setu: {acc:.4f}")

print("\n=============================================")
print(f"PROSEČNA TAČNOST LSTM MODELA: {np.mean(all_fold_accuracies):.4f}")
print("=============================================\n")


=== ZAPOČINJEMO TRENING LSTM MREŽE ===
Fold 1 Završen | Trenutni Loss: 0.1001 | Tačnost na test setu: 0.9655
Fold 2 Završen | Trenutni Loss: 0.0622 | Tačnost na test setu: 0.7586
Fold 3 Završen | Trenutni Loss: 0.0680 | Tačnost na test setu: 0.8276
Fold 4 Završen | Trenutni Loss: 1.0669 | Tačnost na test setu: 0.9655
Fold 5 Završen | Trenutni Loss: 0.1876 | Tačnost na test setu: 0.9643

PROSEČNA TAČNOST LSTM MODELA: 0.8963



In [14]:
#LSTM final trening i cuvanje tezina
print("=== TRENING FINALNOG LSTM MODELA NA SVIM PODACIMA ===")
final_model = GlaucomaLSTM(input_size=len(features_list), hidden_size=32, num_layers=2)
final_optimizer = optim.Adam(final_model.parameters(), lr=0.005)
full_loader = DataLoader(TensorDataset(torch.tensor(X_3D), torch.tensor(y_3D).unsqueeze(1)), batch_size=16, shuffle=True)

final_model.train()
for epoch in range(40):
    for batch_X, batch_y in full_loader:
        final_optimizer.zero_grad()
        outputs = final_model(batch_X)
        loss = final_model_loss = criterion(outputs, batch_y)
        loss.backward()
        final_optimizer.step()

# Čuvanje parametara (težina) modela
# PyTorch ne čuva cijeli model kao joblib, već snima "state_dict" (mapu težina)
torch.save(final_model.state_dict(), "glaucoma_lstm_weights.pth")
print("\nUspešno sačuvano! Fajl 'glaucoma_lstm_weights.pth' sadrži istrenirane težine neuronske mreže.")

=== TRENING FINALNOG LSTM MODELA NA SVIM PODACIMA ===

Uspešno sačuvano! Fajl 'glaucoma_lstm_weights.pth' sadrži istrenirane težine neuronske mreže.


# XGBoost - not good

In [ ]:
#XGBoost ucitavanje

# 2. Ručno preimenovanje ključnih kolona u Baseline
base_cols = list(df_base.columns)
base_cols[7] = 'Progression_PLR2'
base_cols[8] = 'Progression_PLR3'
base_cols[9] = 'Progression_MD'
base_cols[11] = 'RNFL_Mean'
base_cols[12] = 'RNFL_S'
base_cols[13] = 'RNFL_N'
base_cols[14] = 'RNFL_I'
base_cols[15] = 'RNFL_T'

# Preimenovanje VF kolona u Baseline (od indeksa 19 do kraja)
vf_count = 1
for i in range(19, len(base_cols)):
    base_cols[i] = f'VF_{vf_count}'
    vf_count += 1
df_base.columns = base_cols

# 3. Preimenovanje VF kolona u Follow-up (od indeksa 8 do kraja)
follow_cols = list(df_follow.columns)
vf_count = 1
for i in range(8, len(follow_cols)):
    follow_cols[i] = f'VF_{vf_count}'
    vf_count += 1
df_follow.columns = follow_cols

# 4. Standardizacija Baseline-a da liči na Follow-up kako bismo ih spojili
df_base['Visit Number'] = 0.0
df_base['Interval Years'] = 0.0

# === IZMJENA 1: Dodate RNFL kolone u statičke podatke o pacijentu ===
df_patients_static = df_base[[
    'Subject Number', 'Laterality', 'Age', 'Gender', 'CCT', 'Category of Glaucoma', 'Progression_MD',
    'RNFL_S', 'RNFL_N', 'RNFL_I', 'RNFL_T'
]]

# 5. Priprema i spajanje svih posjeta u jedan hronološki niz
df_base_features = df_base[['Subject Number', 'Laterality', 'Visit Number', 'Interval Years', 'IOP'] + [f'VF_{i}' for i in range(1, 62)]]
df_follow_features = df_follow[['Subject Number', 'Laterality', 'Visit Number', 'Interval Years', 'IOP'] + [f'VF_{i}' for i in range(1, 62)]]

# Spajamo ih zajedno
df_all_visits = pd.concat([df_base_features, df_follow_features], axis=0)
df_all_visits = df_all_visits.sort_values(by=['Subject Number', 'Laterality', 'Visit Number']).reset_index(drop=True)

# Zamjena "/" sa NaN i popunjavanje ako negdje fali IOP
df_all_visits = df_all_visits.replace('/', np.nan)
df_all_visits['IOP'] = df_all_visits['IOP'].astype(float)
df_all_visits['IOP'] = df_all_visits.groupby(['Subject Number', 'Laterality'])['IOP'].transform(lambda x: x.fillna(method='ffill'))

# Kreiranje istorijskih (Lag) featura
df_all_visits['prev_IOP'] = df_all_visits.groupby(['Subject Number', 'Laterality'])['IOP'].shift(1)
df_all_visits['IOP_change'] = df_all_visits['IOP'] - df_all_visits['prev_IOP']

# Izračunamo srednju vrijednost vidnog polja (Mean VF)
vf_cols = [f'VF_{i}' for i in range(1, 62)]
df_all_visits[vf_cols] = df_all_visits[vf_cols].astype(float)
df_all_visits['VF_mean'] = df_all_visits[vf_cols].mean(axis=1)

# Uzimamo zadnju poznatu posjetu za svakog pacijenta
df_last_visit = df_all_visits.groupby(['Subject Number', 'Laterality']).last().reset_index()

# Spajamo sa statičkim podacima (sada povlači i RNFL_S, N, I, T koji su spojeni na osnovu Subject Number i Laterality)
df_model_data = pd.merge(df_last_visit, df_patients_static, on=['Subject Number', 'Laterality'], suffixes=('', '_static'))

# Pretvaranje kategoričkih varijabli u brojeve
df_model_data['Gender'] = df_model_data['Gender'].map({'M': 1, 'F': 0})
df_model_data['Category of Glaucoma'] = df_model_data['Category of Glaucoma'].map({'OAG': 1, 'ACG': 0})
df_model_data['Progression_MD'] = df_model_data['Progression_MD'].astype(int)

# Konvertujemo RNFL kolone u float tip i popunjavamo ako ima praznih vrijednosti
rnfl_cols = ['RNFL_S', 'RNFL_N', 'RNFL_I', 'RNFL_T']
df_model_data[rnfl_cols] = df_model_data[rnfl_cols].replace('/', np.nan).astype(float)

/tmp/ipykernel_6009/6506309.py:51: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_all_visits['IOP'] = df_all_visits.groupby(['Subject Number', 'Laterality'])['IOP'].transform(lambda x: x.fillna(method='ffill'))
/tmp/ipykernel_6009/6506309.py:75: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_model_data[rnfl_cols] = df_model_data[rnfl_cols].replace('/', np.nan).astype(float)


In [ ]:
#XGBoost test?
features = [
    'Age', 'Gender', 'CCT', 'Category of Glaucoma',
    'IOP', 'prev_IOP', 'IOP_change', 'VF_mean',
    'RNFL_S', 'RNFL_N', 'RNFL_I', 'RNFL_T'
]

X = df_model_data[features].fillna(0)
y = df_model_data['Progression_MD']
groups = df_model_data['Subject Number']

# Trening i validacija (GroupKFold)
gkf = GroupKFold(n_splits=5)

all_accuracies = []
for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    fold_model = XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.05,
        random_state=42,
        eval_metric='logloss'
    )

    fold_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=True)
    preds = fold_model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    all_accuracies.append(acc)
    print(f"Fold {fold + 1} Tačnost: {acc:.4f}")

print(f"\nProsečna tačnost modela na osnovu cross-validacije: {np.mean(all_accuracies):.4f}\n")


[0]	validation_0-logloss:0.30989
[1]	validation_0-logloss:0.30911
[2]	validation_0-logloss:0.30697
[3]	validation_0-logloss:0.30498
[4]	validation_0-logloss:0.30444
[5]	validation_0-logloss:0.30487
[6]	validation_0-logloss:0.30442
[7]	validation_0-logloss:0.30552
[8]	validation_0-logloss:0.30605
[9]	validation_0-logloss:0.30649
[10]	validation_0-logloss:0.30598
[11]	validation_0-logloss:0.30655
[12]	validation_0-logloss:0.30679
[13]	validation_0-logloss:0.30685
[14]	validation_0-logloss:0.30693
[15]	validation_0-logloss:0.30654
[16]	validation_0-logloss:0.30685
[17]	validation_0-logloss:0.30661
[18]	validation_0-logloss:0.30506
[19]	validation_0-logloss:0.30506
[20]	validation_0-logloss:0.30646
[21]	validation_0-logloss:0.30767
[22]	validation_0-logloss:0.30861
[23]	validation_0-logloss:0.30880
[24]	validation_0-logloss:0.30946
[25]	validation_0-logloss:0.30760
[26]	validation_0-logloss:0.30892
[27]	validation_0-logloss:0.30881
[28]	validation_0-logloss:0.30758
[29]	validation_0-loglos

['model_features.pkl']

In [ ]:
#XGBoost ucitavanje modela
model = joblib.load("glaucoma_progression_xgb_model.pkl")

In [ ]:
#XGBoost skladistenje modela
final_model = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    random_state=42,
    eval_metric='logloss'
)

final_model.fit(X, y, verbose=False)

model_filename = "glaucoma_progression_xgb_model.pkl"
joblib.dump(final_model, model_filename)

joblib.dump(features, "model_features.pkl")

['model_features.pkl']